In [1]:
!pip -q install "dython==0.7.5" --no-deps


In [2]:
!pip -q install scikit-learn scipy matplotlib seaborn tqdm


In [3]:
import numpy, pandas, dython
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("dython:", dython.__version__)


numpy: 2.0.2
pandas: 2.2.2
dython: 0.7.5


In [4]:
!pip -q install torch


In [1]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = "/content/drive/MyDrive/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main/data/diabetes_012_health_indicators_BRFSS2015.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main/data/diabetes_012_health_indicators_BRFSS2015.csv")
df.head()


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [7]:
!pip -q install opacus


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 8.9 MB/s eta 0:00:00


In [8]:
from opacus import PrivacyEngine


In [9]:
import sys, os

REPO_DIR = "/content/drive/MyDrive/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main"   # change if different
sys.path.insert(0, REPO_DIR)

from model.pipeline.data_preparation import DataPrep
from model.synthesizer.ctabgan_synthesizer import CTABGANSynthesizer
print("Imports OK")

Imports OK


In [10]:
import numpy as np
import pandas as pd

from model.pipeline.data_preparation import DataPrep
from model.synthesizer.ctabgan_synthesizer import CTABGANSynthesizer

# -----------------------------
# 1) Read CSV robustly
# -----------------------------
df = pd.read_csv("/content/drive/MyDrive/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main/data/diabetes_012_health_indicators_BRFSS2015.csv")

# Drop completely empty columns
df = df.dropna(axis=1, how="all")

# Drop constant columns (CTGAN-like models often struggle with them)
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df_model = df.drop(columns=constant_cols) if constant_cols else df.copy()

# -----------------------------
# 2) Auto-convert datetime-like object columns to timestamps
# -----------------------------
datetime_cols = []
for c in df_model.columns:
    if df_model[c].dtype == "object":
        parsed = pd.to_datetime(df_model[c], errors="coerce", infer_datetime_format=True)
        if parsed.notna().mean() >= 0.90:  # 90% parseable -> treat as datetime
            # seconds since epoch (float)
            df_model[c] = (parsed.view("int64") // 10**9).astype("float64")
            datetime_cols.append(c)

# -----------------------------
# 3) Auto-detect categorical + integer columns (no manual inputs)
# -----------------------------
n = len(df_model)

# Start with obvious categorical
categorical_cols = df_model.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

# Treat low-cardinality numeric as categorical
for c in df_model.columns:
    if c not in categorical_cols and pd.api.types.is_numeric_dtype(df_model[c]):
        nunique = df_model[c].nunique(dropna=True)
        if nunique <= 20 or (n > 0 and nunique / n <= 0.02):  # <=20 unique or <=2% unique ratio
            categorical_cols.append(c)

# Integer columns: numeric columns that are "integer-like"
integer_cols = []
for c in df_model.columns:
    if c in categorical_cols:
        continue
    if pd.api.types.is_numeric_dtype(df_model[c]):
        s = df_model[c].dropna()
        if len(s) > 0 and np.all(np.isclose(s.values, np.round(s.values))):
            integer_cols.append(c)

# We’ll keep these empty/generic
log_cols = []
mixed_cols = {}               # DataPrep will auto-handle missing numeric columns internally
general_cols = []             # keep empty for generic behavior
non_categorical_cols = []     # keep empty

# Unsupervised / generic (no target)
problem_type = {None: None}

# -----------------------------
# 4) Train + Sample
# -----------------------------
dp = DataPrep(
    raw_df=df_model,
    categorical=categorical_cols,
    log=log_cols,
    mixed=mixed_cols,
    general=general_cols,
    non_categorical=non_categorical_cols,
    integer=integer_cols,
    type=problem_type,
    test_ratio=0.0,   # unsupervised; keep all rows
)

# Tune these if you want faster/slower
synth = CTABGANSynthesizer(epochs=30, batch_size=512)  # try epochs=10 for quick test

synth.fit(
    train_data=dp.df,
    categorical=dp.column_types["categorical"],
    mixed=dp.column_types["mixed"],
    general=dp.column_types["general"],
    non_categorical=dp.column_types["non_categorical"],
    type=problem_type
)

sample = synth.sample(len(df_model))
syn = dp.inverse_prep(sample)

# Add constant columns back exactly as original
for c in constant_cols:
    syn[c] = df[c].mode(dropna=False).iloc[0] if len(df) else df[c].iloc[0]

# Reorder columns to match original CSV
syn = syn[df.columns]

syn.to_csv("synthetic.csv", index=False)

print("Done ✅")
print("Original shape:", df.shape)
print("Model shape   :", df_model.shape, "(after dropping constants/empty)")
print("Synthetic shape:", syn.shape)
print("Auto categorical cols:", len(categorical_cols))
print("Auto integer cols:", len(integer_cols))
syn.head()


100%|██████████| 30/30 [49:31<00:00, 99.05s/it]


Done ✅
Original shape: (253680, 22)
Model shape   : (253680, 22) (after dropping constants/empty)
Synthetic shape: (253680, 22)
Auto categorical cols: 22
Auto integer cols: 0


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,0.0,3.0,6.0,7.0
1,0.0,0.0,1.0,1.0,25.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,8.0,6.0,7.0
2,0.0,0.0,0.0,1.0,25.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,12.0,4.0,4.0
3,0.0,0.0,0.0,1.0,32.0,0.0,0.0,0.0,1.0,1.0,...,1.0,1.0,2.0,3.0,4.0,0.0,0.0,8.0,5.0,8.0
4,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,5.0,10.0,0.0,0.0,3.0,4.0,5.0


In [11]:
from google.colab import drive
drive.mount("/content/drive")

OUT_PATH = "/content/drive/MyDrive/CTAB-GAN-Plus-outputs/diabetes_synthetic.csv"

import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

syn.to_csv(OUT_PATH, index=False)
print("Saved to:", OUT_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/CTAB-GAN-Plus-outputs/diabetes_synthetic.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from scipy.stats import ks_2samp, entropy, spearmanr
from sklearn.metrics import normalized_mutual_info_score

# =========================
# CONFIG (edit these)
# =========================
REAL_PATH = "/content/drive/MyDrive/CTAB-GAN-Plus-main/CTAB-GAN-Plus-main/data/diabetes_012_health_indicators_BRFSS2015.csv"
SYN_PATH  = "/content/drive/MyDrive/CTAB-GAN-Plus-outputs/diabetes_synthetic.csv"

# If your dataset has a label column you want excluded from distribution metrics:
TARGET_COL = "Target"   # set to None if you don't have one

# High-cardinality categorical guard (prevents giant one-hot)
MAX_OHE_LEVELS = 100     # categoricals with > this unique values won't be one-hot encoded in privacy DCR calc

# Privacy DCR speed cap (optional)
MAX_ROWS_FOR_DCR = 8000  # subsample for KNN distance if needed (set None for no cap)

# =========================
# Helpers
# =========================
def combine_date_time(df: pd.DataFrame) -> pd.DataFrame:
    """If Date/Time exist, create numeric DateTime seconds and drop Date/Time."""
    out = df.copy()
    if ("Date" in out.columns) or ("Time" in out.columns):
        date_part = out["Date"].astype(str) if "Date" in out.columns else "1970-01-01"
        time_part = out["Time"].astype(str) if "Time" in out.columns else "00:00:00"
        dt = pd.to_datetime(date_part + " " + time_part, errors="coerce")
        # seconds since epoch; NaT -> nan
        dt_int = dt.astype("datetime64[ns]").astype("int64").astype("float64")
        dt_int[dt.isna().to_numpy()] = np.nan
        out["DateTime"] = dt_int / 1e9
        for c in ["Date", "Time"]:
            if c in out.columns:
                out = out.drop(columns=[c])
    return out

def infer_cat_num(df: pd.DataFrame, cols):
    """Infer categorical vs numeric columns generically."""
    cat_cols = []
    for c in cols:
        s = df[c]
        if s.dtype.name in ["object", "category", "bool"]:
            cat_cols.append(c)
        else:
            # treat low-cardinality numeric as categorical (often IDs/codes)
            nunique = s.nunique(dropna=True)
            n = len(s)
            if nunique <= 20 or (n > 0 and nunique / max(1, n) <= 0.02):
                # only if it's integer-like or small set
                cat_cols.append(c)
    num_cols = [c for c in cols if c not in cat_cols]
    return cat_cols, num_cols

# ---------- Quality metrics ----------
def js_divergence(p, q, base=np.e):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    p = p / (p.sum() + 1e-12); q = q / (q.sum() + 1e-12)
    m = 0.5 * (p + q)
    return 0.5 * entropy(p, m, base=base) + 0.5 * entropy(q, m, base=base)

def hist_jsd(xr, xs, bins=30, smooth=1e-9):
    xr = np.asarray(xr); xs = np.asarray(xs)
    lo = np.nanmin(np.concatenate([xr, xs])); hi = np.nanmax(np.concatenate([xr, xs]))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = 0.0, 1.0 if lo == hi else (lo, hi)
    hr, _ = np.histogram(xr, bins=bins, range=(lo, hi))
    hs, _ = np.histogram(xs, bins=bins, range=(lo, hi))
    pr = (hr + smooth) / (hr.sum() + smooth*bins)
    ps = (hs + smooth) / (hs.sum() + smooth*bins)
    return float(js_divergence(pr, ps))

def tvd_from_counts(p, q):
    keys = sorted(set(p).union(q), key=str)
    pv = np.array([p.get(k, 0.0) for k in keys], dtype=float)
    qv = np.array([q.get(k, 0.0) for k in keys], dtype=float)
    return 0.5 * np.abs(pv - qv).sum()

def spearman_matrix(df, cols):
    m = len(cols)
    R = np.ones((m, m), dtype=float) * np.nan
    for i, a in enumerate(cols):
        R[i, i] = 1.0
        for j in range(i+1, m):
            b = cols[j]
            rho, _ = spearmanr(df[a], df[b], nan_policy="omit")
            R[i, j] = R[j, i] = rho
    return R

def nmi_matrix(df, cols):
    m = len(cols)
    M = np.ones((m, m), dtype=float) * np.nan
    for i, a in enumerate(cols):
        M[i, i] = 1.0
        for j in range(i+1, m):
            b = cols[j]
            M[i, j] = M[j, i] = normalized_mutual_info_score(df[a].astype(str), df[b].astype(str))
    return M

def score_quality_Q(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col=None):
    cols = list(real_df.columns)
    if target_col and target_col in cols:
        cols = [c for c in cols if c != target_col]

    cat_cols, num_cols = infer_cat_num(real_df, cols)

    # --- D: numeric (KS, JSD, range coverage) ---
    D_parts = []
    for c in num_cols:
        xr = pd.to_numeric(real_df[c], errors="coerce").dropna().values
        xs = pd.to_numeric(syn_df[c],  errors="coerce").dropna().values
        if xr.size and xs.size:
            ks = ks_2samp(xr, xs).statistic
            jsd = hist_jsd(xr, xs)
            lo, hi = np.nanmin(xr), np.nanmax(xr)
            rc = float(np.mean((xs >= lo) & (xs <= hi))) if np.isfinite(lo) and np.isfinite(hi) else np.nan
            D_parts.append(np.nanmean([1-ks, 1-jsd, rc]))
    D_num = float(np.nanmean(D_parts)) if D_parts else np.nan

    # --- D: categorical (1 - TVD) ---
    D_cat = np.nan
    if cat_cols:
        tvds = []
        rr = real_df[cat_cols].astype(str)
        ss = syn_df[cat_cols].astype(str)
        for c in cat_cols:
            pr = rr[c].value_counts(normalize=True).to_dict()
            ps = ss[c].value_counts(normalize=True).to_dict()
            tvds.append(1.0 - tvd_from_counts(pr, ps))
        D_cat = float(np.mean(tvds)) if tvds else np.nan

    # --- C: numeric (Spearman matrix similarity) ---
    C_num = np.nan
    if len(num_cols) >= 2:
        Rr = spearman_matrix(real_df[num_cols].apply(pd.to_numeric, errors="coerce"), num_cols)
        Rs = spearman_matrix(syn_df[num_cols].apply(pd.to_numeric, errors="coerce"),  num_cols)
        mask = np.isfinite(Rr) & np.isfinite(Rs)
        C_num = float(1.0 - np.nanmean(np.abs(Rr[mask] - Rs[mask])))

    # --- C: categorical (NMI matrix similarity) ---
    C_cat = np.nan
    if len(cat_cols) >= 2:
        rr = real_df[cat_cols].astype(str)
        ss = syn_df[cat_cols].astype(str)
        Mr = nmi_matrix(rr, cat_cols)
        Ms = nmi_matrix(ss, cat_cols)
        mask = np.isfinite(Mr) & np.isfinite(Ms)
        C_cat = float(1.0 - np.nanmean(np.abs(Mr[mask] - Ms[mask])))

    # --- V: categorical coverage (real categories present in synthetic) ---
    V = np.nan
    if cat_cols:
        rr = real_df[cat_cols].astype(str)
        ss = syn_df[cat_cols].astype(str)
        cov = []
        for c in cat_cols:
            real_cats = set(rr[c].dropna().unique())
            syn_cats  = set(ss[c].dropna().unique())
            cov.append(len(real_cats & syn_cats) / max(1, len(real_cats)))
        V = float(np.mean(cov)) if cov else np.nan

    parts = [x for x in [D_num, D_cat, C_num, C_cat, V] if not np.isnan(x)]
    Q = float(np.mean(parts)) if parts else 0.0

    return {
        "Q": Q,
        "D_num": D_num,
        "D_cat": D_cat,
        "C_num": C_num,
        "C_cat": C_cat,
        "V": V,
        "num_cols": len(num_cols),
        "cat_cols": len(cat_cols),
    }

# ---------- Privacy metrics ----------
def score_privacy_P(real_df: pd.DataFrame, syn_df: pd.DataFrame, target_col=None):
    cols = list(real_df.columns)
    if target_col and target_col in cols:
        cols = [c for c in cols if c != target_col]

    # infer cat/num for privacy encoding
    cat_cols, num_cols = infer_cat_num(real_df, cols)

    # High-cardinality categorical guard for one-hot
    cat_ok = []
    cat_hi = []
    for c in cat_cols:
        if real_df[c].nunique(dropna=True) <= MAX_OHE_LEVELS:
            cat_ok.append(c)
        else:
            cat_hi.append(c)

    # Build numeric matrix for high-card categoricals via factorize (keeps KNN stable)
    def factorize_cols(df, cols_):
        out = pd.DataFrame(index=df.index)
        for c in cols_:
            codes, _ = pd.factorize(df[c].astype("string").fillna("<NA>"), sort=False)
            out[c] = codes.astype("float64")
        return out

    real_num = real_df[num_cols].apply(pd.to_numeric, errors="coerce")
    syn_num  = syn_df[num_cols].apply(pd.to_numeric, errors="coerce")

    # median impute numeric using real medians
    med = real_num.median(numeric_only=True)
    real_num = real_num.fillna(med)
    syn_num  = syn_num.fillna(med)

    real_hi = factorize_cols(real_df, cat_hi) if cat_hi else pd.DataFrame(index=real_df.index)
    syn_hi  = factorize_cols(syn_df,  cat_hi) if cat_hi else pd.DataFrame(index=syn_df.index)

    # One-hot only safe categoricals
    def make_ohe():
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=False)

    enc = ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", make_ohe(),    cat_ok),
    ])

    Xr_base = enc.fit_transform(pd.concat([real_num, real_df[cat_ok].astype(str)], axis=1) if cat_ok else real_num)
    Xs_base = enc.transform(pd.concat([syn_num,  syn_df[cat_ok].astype(str)],  axis=1) if cat_ok else syn_num)

    # append factorized high-card cols as numeric
    if cat_hi:
        X_real = np.hstack([Xr_base, real_hi.to_numpy(dtype=np.float64)])
        X_syn  = np.hstack([Xs_base, syn_hi.to_numpy(dtype=np.float64)])
    else:
        X_real = Xr_base
        X_syn  = Xs_base

    X_real = np.asarray(X_real, dtype=np.float32)
    X_syn  = np.asarray(X_syn,  dtype=np.float32)

    # Optional subsampling for speed
    if MAX_ROWS_FOR_DCR and len(X_real) > MAX_ROWS_FOR_DCR:
        idx = np.random.RandomState(42).choice(len(X_real), MAX_ROWS_FOR_DCR, replace=False)
        X_real_sub = X_real[idx]
    else:
        X_real_sub = X_real

    if MAX_ROWS_FOR_DCR and len(X_syn) > MAX_ROWS_FOR_DCR:
        idx = np.random.RandomState(42).choice(len(X_syn), MAX_ROWS_FOR_DCR, replace=False)
        X_syn_sub = X_syn[idx]
    else:
        X_syn_sub = X_syn

    # DCR (distance to closest real)
    nn = NearestNeighbors(n_neighbors=1).fit(X_real_sub)
    dists, _ = nn.kneighbors(X_syn_sub)
    dists = dists.ravel()
    DCR_mean = float(np.mean(dists))

    # QΔ (quantile drift on numeric)
    quantiles = np.linspace(0.05, 0.95, 19)
    qd_vals = []
    for c in num_cols:
        xr = real_num[c].to_numpy()
        xs = syn_num[c].to_numpy()
        if xr.size and xs.size:
            qr = np.quantile(xr, quantiles)
            qs = np.quantile(xs, quantiles)
            qd_vals.append(float(np.mean(np.abs(qr - qs))))
    Q_delta = float(np.mean(qd_vals)) if qd_vals else float("nan")

    # I (dup within synth + overlap between)
    def norm_df(df):
        out = df[cols].copy()
        for c in cols:
            if np.issubdtype(out[c].dtype, np.number):
                out[c] = pd.to_numeric(out[c], errors="coerce").round(6)
            else:
                out[c] = out[c].astype(str).str.strip()
        return out

    real_norm = norm_df(real_df)
    syn_norm  = norm_df(syn_df)

    dup_within_synth = float(1.0 - len(syn_norm.drop_duplicates()) / max(1, len(syn_norm)))

    set_real = set(map(tuple, real_norm.to_numpy()))
    set_syn  = set(map(tuple, syn_norm.to_numpy()))
    overlap_between = float(len(set_real & set_syn) / max(1, len(syn_norm)))

    # Map to [0,1] scores (same idea as your weights)
    def normalize01(x, lo=0.0, hi=1.0):
        if np.isnan(x): return np.nan
        if hi == lo: return 0.0
        v = (x - lo) / (hi - lo)
        return float(max(0.0, min(1.0, v)))

    def inv01(x, cap=1.0):
        if np.isnan(x): return np.nan
        return float(max(0.0, min(1.0, 1.0 - x / cap)))

    d95 = float(np.percentile(dists, 95)) if len(dists) else 1.0
    DCR_score = normalize01(DCR_mean, lo=0.0, hi=d95 if d95 > 0 else 1.0)

    Qd_cap = float(np.nanmedian(qd_vals)) * 4 if qd_vals else (abs(Q_delta) + 1e-6)
    Qdelta_score = inv01(Q_delta, cap=Qd_cap if Qd_cap > 0 else 1.0)

    I_combined = 0.5 * (dup_within_synth + overlap_between)
    I_score = inv01(I_combined, cap=0.10)

    P = 0.5 * DCR_score + 0.3 * Qdelta_score + 0.2 * I_score

    return {
        "P": float(P),
        "DCR_mean": DCR_mean,
        "DCR_score": DCR_score,
        "Q_delta": Q_delta,
        "Qdelta_score": Qdelta_score,
        "dup_within_synth": dup_within_synth,
        "overlap_between": overlap_between,
        "I_score": I_score,
        "cat_ohe_used": len(cat_ok),
        "cat_hi_card": len(cat_hi),
    }

# =========================
# RUN EVALUATION
# =========================
real_df = pd.read_csv(REAL_PATH, low_memory=False)
syn_df  = pd.read_csv(SYN_PATH,  low_memory=False)

# Align columns + handle Date/Time
all_cols = sorted(set(real_df.columns).union(set(syn_df.columns)))
real_df = combine_date_time(real_df.reindex(columns=all_cols))
syn_df  = combine_date_time(syn_df.reindex(columns=all_cols))

q = score_quality_Q(real_df, syn_df, target_col=TARGET_COL)
p = score_privacy_P(real_df, syn_df, target_col=TARGET_COL)

print("\n===== QUALITY (Q) =====")
for k in ["Q","D_num","D_cat","C_num","C_cat","V","num_cols","cat_cols"]:
    print(f"{k:>10}: {q.get(k)}")

print("\n===== PRIVACY (P) =====")
for k in ["P","DCR_mean","DCR_score","Q_delta","Qdelta_score","dup_within_synth","overlap_between","I_score","cat_ohe_used","cat_hi_card"]:
    print(f"{k:>18}: {p.get(k)}")

# Optional: return a single dict you can save as JSON
results = {"quality": q, "privacy": p}
results



===== QUALITY (Q) =====
         Q: 0.9855322747004428
     D_num: nan
     D_cat: 0.9611002408187838
     C_num: nan
     C_cat: 0.9976610854470465
         V: 0.9978354978354979
  num_cols: 0
  cat_cols: 22

===== PRIVACY (P) =====
                 P: nan
          DCR_mean: 2.4258495105952025
         DCR_score: 0.7671209621257998
           Q_delta: nan
      Qdelta_score: nan
  dup_within_synth: 0.058987701040681184
   overlap_between: 0.05076474298328603
           I_score: 0.4512377798801639
      cat_ohe_used: 22
       cat_hi_card: 0


{'quality': {'Q': 0.9855322747004428,
  'D_num': nan,
  'D_cat': 0.9611002408187838,
  'C_num': nan,
  'C_cat': 0.9976610854470465,
  'V': 0.9978354978354979,
  'num_cols': 0,
  'cat_cols': 22},
 'privacy': {'P': nan,
  'DCR_mean': 2.4258495105952025,
  'DCR_score': 0.7671209621257998,
  'Q_delta': nan,
  'Qdelta_score': nan,
  'dup_within_synth': 0.058987701040681184,
  'overlap_between': 0.05076474298328603,
  'I_score': 0.4512377798801639,
  'cat_ohe_used': 22,
  'cat_hi_card': 0}}